In [ ]:
import nest_asyncio
nest_asyncio.apply()
from djitellopy import Tello
import cv2, math, time
import numpy as np
import asyncio
import bleak
import device_model
from collections import deque

In [ ]:
tello: Tello | None = None
imu_mac = "F2:65:C4:C2:39:26" #device address
imu_buffer = deque(maxlen=20)  
boolean = 0

In [ ]:
#IMU data update

prev_time = time.time()
deltat = 0
def updateData(DeviceModel):
    data = DeviceModel.deviceData
    current_time = time.time()
    imu_buffer.append((time.time(), data))


In [ ]:
#reading 1 sec of IMU data

def get_last_second(t_len):
    now = time.time()
    one_sec_data = [d for t, d in imu_buffer if now - t <= float(t_len)]

    if not one_sec_data:
        return None

    compressed = {}
    keys = one_sec_data[0].keys()
    for k in keys:
        vals = [d[k] for d in one_sec_data if d[k] is not None]
        if vals:
            compressed[k] = sum(vals) / len(vals)
    return compressed

In [ ]:
#gesture controller

def takeOff_land(cd):
    global boolean, tello
    
    if tello is None:
        return
    
    prevAngZ = 0
    AngX = cd.get("AngX")
    AngY = cd.get("AngY")
    AngZ = cd.get("AngZ")
    
    if boolean == 0:
        prevAngZ = AngZ
    
    if boolean == 0 and AngX > 90.0:
        tello.takeoff()
        boolean = 1
        return

    if boolean == 1 and AngX < -90.0:
        tello.land()
        boolean = 0
        return
    
    if boolean == 1 and AngZ - prevAngZ < 20:
        tello.rotate_clockwise(20)
        return
        
    if boolean == 1 and AngZ - prevAngZ > 20:
        tello.rotate_counter_clockwise(20)
        return
    
    if boolean == 1 and AngX < -30.0:
        tello.move_forward(30)
        return
        
    if boolean == 1 and AngY > 50.0:
        tello.move_right(30)
        return
        
    if boolean == 1 and AngY < -50.0:
        tello.move_left(30)
        return
        
    if boolean == 1 and 90.0 > AngX > 30.0:
        tello.move_back(30) 
        return
        


In [ ]:
#connecting to IMU

async def connect_device(imu_mac):
    global imu
    imu = device_model.DeviceModel("MyBle5.0", imu_mac, updateData)
    devices = await bleak.BleakScanner.discover()
    found = False
    for d in devices:
        if d.address == imu_mac:
            found = True
            break
    
    if not found:
        print("IMU not found")
        return
    
    await imu.openDevice()


In [ ]:
#IMU queue
async def imu_loop():
    last_print = time.time()
    last_seen = None

    while True:
        if imu_buffer:
            ts, data = imu_buffer[-1]
            if ts != last_seen:
                last_seen = ts

        if time.time() - last_print >= 0.5:
            cd = get_last_second(0.5)
            
            if cd is not None:
                takeOff_land(cd)    

            last_print = time.time()
        

        await asyncio.sleep(0.01)

In [ ]:
#Main drone function

async def run_tello():
    global tello
    tello = Tello()
    
    await asyncio.to_thread(tello.connect)
    print("Tello connected.")
    
    try:
        battery = await asyncio.to_thread(tello.get_battery)
        print("Tello battery:", battery)
    except Exception as e:
        print("Could not read battery:", e)
    
    try:
        while True:
            await asyncio.sleep(1.0)
    finally:
        try:
            await asyncio.to_thread(tello.end)
        except Exception:
            pass

In [ ]:
#Wait for first IMU signal

async def wait_for_first_imu(timeout: float = 10.0) -> bool:
    start = time.time()
    print(f"Waiting for first IMU sample (timeout={timeout}s)...")
    while time.time() - start < timeout:
        if imu_buffer:
            ts, data = imu_buffer[-1]
            print("First IMU sample received at", ts, ":", data)
            return True
        await asyncio.sleep(0.05)

    print("No IMU data received within timeout.")
    return False

In [ ]:
async def main_loop():

    ble_task = asyncio.create_task(connect_device(imu_mac))
    imu_task = asyncio.create_task(imu_loop())
    tello_task = None

    try:
        got_imu = await wait_for_first_imu(timeout=20.0)

        if got_imu:
            print("starting Tello connection")
            tello_task = asyncio.create_task(run_tello())


        # Keep running whatever tasks exist
        tasks = [ble_task, imu_task]
        if tello_task is not None:
            tasks.append(tello_task)

        await asyncio.gather(*tasks)

    finally:
        # Cleanup on exit
        global imu
        if imu is not None:
            try:
                imu.closeDevice()
            except Exception:
                pass

        for task in (ble_task, imu_task, tello_task):
            if task and not task.done():
                task.cancel()

In [ ]:
await main_loop()